# AGHCT - Medical Image Segmentation
**Session kesilirse Hücre 1 ve Hücre 3'ü tekrar çalıştır, kaldığı yerden devam eder.**

In [ ]:
#@title 1. SETUP

from google.colab import drive
drive.mount('/content/drive')

import torch, sys, os, shutil, yaml, functools
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# torch.load fix
_orig = torch.load
@functools.wraps(_orig)
def _pl(*a, **kw):
    kw.setdefault('weights_only', False)
    return _orig(*a, **kw)
torch.load = _pl

# Path'ler - hepsi Drive kokunde
CODE_DIR = '/content/drive/MyDrive/aghct'
CKPT_DIR = '/content/drive/MyDrive/checkpoints'
RESULTS_DIR = '/content/drive/MyDrive/results'
DATASET_ZIP = '/content/drive/MyDrive/archive (1).zip'
DATASET_DIR = '/content/isic2018'
CONFIG_PATH = '/content/configs/config.yaml'

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Kod kontrol
assert os.path.exists(CODE_DIR), f'Kod bulunamadi: {CODE_DIR}'
sys.path.insert(0, CODE_DIR)
os.chdir(CODE_DIR)
print(f'Kod: OK')

# Kutuphaneler
!pip install -q albumentations medpy einops timm 2>/dev/null
print('Kutuphaneler: OK')

# Dataset - her session'da unzip gerekir
if os.path.exists(DATASET_DIR) and len(os.listdir(DATASET_DIR)) > 0:
    print(f'Dataset: zaten mevcut')
else:
    assert os.path.exists(DATASET_ZIP), f'Dataset zip bulunamadi: {DATASET_ZIP}'
    print('Dataset unzip ediliyor...')
    os.makedirs(DATASET_DIR, exist_ok=True)
    !unzip -q "{DATASET_ZIP}" -d {DATASET_DIR}
    print('Dataset: OK')

for item in os.listdir(DATASET_DIR):
    sub = os.path.join(DATASET_DIR, item)
    if os.path.isdir(sub):
        print(f'  {item}: {len(os.listdir(sub))} dosya')

# Config
os.makedirs('/content/configs', exist_ok=True)
shutil.copy(f'{CODE_DIR}/configs/config.yaml', CONFIG_PATH)
with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)
config['dataset']['isic']['root'] = DATASET_DIR
config['dataset']['isic']['batch_size'] = 16
config['kaggle'] = {'num_workers': 2, 'input_dir': '/content', 'working_dir': '/content'}
config['checkpoint'] = config.get('checkpoint', {})
config['checkpoint']['save_dir'] = CKPT_DIR
config['checkpoint']['save_every'] = 5
config['checkpoint']['save_best'] = True
config['checkpoint']['resume'] = True
with open(CONFIG_PATH, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f'Config: OK')
print('\n=== SETUP TAMAMLANDI ===')

In [ ]:
#@title 2. HIZLI TEST

from models.unet import UNet
from models.aghct import AGHCT
from data.isic_dataset import ISICDataset
from data.augmentations import get_train_transforms

x = torch.randn(1, 3, 256, 256)
m1 = UNet()
print(f'UNet:  {x.shape} -> {m1(x).shape} | {sum(p.numel() for p in m1.parameters())/1e6:.1f}M')
m2 = AGHCT(config['model'])
print(f'AGHCT: {x.shape} -> {m2(x).shape} | {sum(p.numel() for p in m2.parameters())/1e6:.1f}M')
ds = ISICDataset(root_dir=DATASET_DIR, split='train', transform=get_train_transforms(256), fold=0)
img, mask = ds[0]
print(f'Dataset: {len(ds)} goruntu | img={img.shape} mask={mask.shape}')
print('Her sey calisiyor!')

In [ ]:
#@title 3. EGITIM - U-Net 2 fold + AGHCT 2 fold, 50 epoch
# Session kesilirse bu hucreyi TEKRAR CALISTIR (once Hucre 1'i calistir)

from train import main as train_main

EPOCHS = 50
results = {}

for model_name in ['unet', 'aghct']:
    for fold in [0, 1]:
        best_path = f'{CKPT_DIR}/{model_name}_isic_fold{fold}_frac1.0_best.pth'
        last_path = f'{CKPT_DIR}/{model_name}_isic_fold{fold}_frac1.0_last.pth'
        
        # Tamamlanmis mi?
        if os.path.exists(best_path):
            ckpt = torch.load(best_path, map_location='cpu')
            done_epoch = ckpt.get('epoch', -1)
            best_dice = ckpt.get('best_dice', 0)
            if done_epoch >= EPOCHS - 2 and best_dice > 0.80:
                print(f'>>> {model_name.upper()} FOLD {fold} TAMAMLANMIS - dice={best_dice:.4f}, ATLANIYOR')
                results[f'{model_name}_fold{fold}'] = best_dice
                continue
        
        # Resume?
        resume_flag = []
        if os.path.exists(last_path):
            ckpt = torch.load(last_path, map_location='cpu')
            last_epoch = ckpt.get('epoch', 0)
            if last_epoch > 0:
                print(f'>>> {model_name.upper()} FOLD {fold} - Epoch {last_epoch} den devam')
                resume_flag = ['--resume']
        
        print(f'\n{"="*60}')
        print(f'{model_name.upper()} FOLD {fold} - {EPOCHS} epoch')
        print(f'{"="*60}\n')
        
        sys.argv = [
            'train.py',
            '--config', CONFIG_PATH,
            '--model', model_name,
            '--dataset', 'isic',
            '--fold', str(fold),
            '--epochs', str(EPOCHS),
        ] + resume_flag
        train_main()
        
        if os.path.exists(best_path):
            ckpt = torch.load(best_path, map_location='cpu')
            dice = ckpt.get('best_dice', 0)
            results[f'{model_name}_fold{fold}'] = dice
            print(f'>>> {model_name.upper()} FOLD {fold} BEST DICE: {dice:.4f}')

print(f'\n{"="*60}')
print('TUM SONUCLAR:')
print(f'{"="*60}')
for k, v in results.items():
    print(f'  {k}: {v:.4f}')

In [ ]:
#@title 4. SONUC TABLOSU
import numpy as np

results = {}
for model_name in ['unet', 'aghct']:
    for fold in [0, 1]:
        path = f'{CKPT_DIR}/{model_name}_isic_fold{fold}_frac1.0_best.pth'
        if os.path.exists(path):
            ckpt = torch.load(path, map_location='cpu')
            results[f'{model_name}_fold{fold}'] = ckpt.get('best_dice', 0)

unet_dices = [v for k, v in results.items() if 'unet' in k]
aghct_dices = [v for k, v in results.items() if 'aghct' in k]

print('='*55)
print('  ISIC 2018 Skin Lesion Segmentation')
print('='*55)
print(f'  {"Model":<20} {"Dice (mean +/- std)":<25} {"Params"}')
print('-'*55)
if unet_dices:
    print(f'  {"U-Net":<20} {np.mean(unet_dices):.4f} +/- {np.std(unet_dices):.4f}        31.04M')
if aghct_dices:
    print(f'  {"AGHCT (Ours)":<20} {np.mean(aghct_dices):.4f} +/- {np.std(aghct_dices):.4f}        24.20M')
print('-'*55)
if unet_dices and aghct_dices:
    diff = np.mean(aghct_dices) - np.mean(unet_dices)
    print(f'\n  Fark: {diff:+.4f} Dice')
print(f'\nDetay:')
for k, v in results.items():
    print(f'  {k}: {v:.4f}')
with open(f'{RESULTS_DIR}/results.txt', 'w') as f:
    for k, v in results.items():
        f.write(f'{k}: {v:.4f}\n')
print(f'Kaydedildi: {RESULTS_DIR}/results.txt')

In [ ]:
#@title 5. GORSEL SONUCLAR
import matplotlib.pyplot as plt
from data.isic_dataset import ISICDataset
from data.augmentations import get_val_transforms
from models.unet import UNet
from models.aghct import AGHCT

device = torch.device('cuda')
val_ds = ISICDataset(root_dir=DATASET_DIR, split='val', transform=get_val_transforms(256), fold=0)

models_loaded = {}
for name in ['unet', 'aghct']:
    bp = f'{CKPT_DIR}/{name}_isic_fold0_frac1.0_best.pth'
    if os.path.exists(bp):
        m = UNet() if name == 'unet' else AGHCT(config['model'])
        ckpt = torch.load(bp, map_location='cpu')
        m.load_state_dict(ckpt['model_state_dict'])
        m.to(device).eval()
        models_loaded[name] = m

ncols = 2 + len(models_loaded)
fig, axes = plt.subplots(5, ncols, figsize=(4*ncols, 20))
mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

for i, idx in enumerate([0, 50, 100, 150, 200]):
    if idx >= len(val_ds): idx = i
    img, mask = val_ds[idx]
    img_show = (img * std + mean).permute(1,2,0).numpy().clip(0,1)
    mask_show = mask.numpy() if mask.ndim == 2 else mask.squeeze().numpy()
    axes[i][0].imshow(img_show)
    axes[i][0].set_title('Input' if i==0 else '')
    axes[i][0].axis('off')
    axes[i][1].imshow(mask_show, cmap='gray')
    axes[i][1].set_title('Ground Truth' if i==0 else '')
    axes[i][1].axis('off')
    col = 2
    for name, m in models_loaded.items():
        with torch.no_grad():
            pred = torch.sigmoid(m(img.unsqueeze(0).to(device)))
            pred = (pred > 0.5).float().cpu().squeeze().numpy()
        axes[i][col].imshow(pred, cmap='gray')
        axes[i][col].set_title(name.upper() if i==0 else '')
        axes[i][col].axis('off')
        col += 1

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/qualitative_results.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Kaydedildi: {RESULTS_DIR}/qualitative_results.png')

In [ ]:
#@title 6. DETAYLI METRIKLER
from torch.utils.data import DataLoader
from utils.metrics import compute_all_metrics, count_parameters
from tqdm import tqdm
import numpy as np

val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2)

print('='*70)
print('  DETAYLI METRIKLER - ISIC 2018')
print('='*70)

for name, m in models_loaded.items():
    ml = {'dice': [], 'iou': [], 'sensitivity': [], 'specificity': []}
    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc=name):
            images = images.to(device)
            masks = masks.to(device).unsqueeze(1) if masks.ndim == 3 else masks.to(device)
            preds = m(images)
            for j in range(images.size(0)):
                met = compute_all_metrics(preds[j:j+1], masks[j:j+1])
                for k in ml:
                    if k in met: ml[k].append(met[k])
    print(f'\n  {name.upper()} ({count_parameters(m):.2f}M):')
    for k, v in ml.items():
        vals = [x for x in v if x != float('inf')]
        if vals: print(f'    {k:>15}: {np.mean(vals):.4f} +/- {np.std(vals):.4f}')

print(f'\nKaydedildi: {RESULTS_DIR}/')